# **Preparing data and Adding Features**

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
import keras

In [43]:
symbol="BTC-USD"
df=pd.DataFrame(data=yf.download(tickers=symbol,period="1mo",interval="5m",auto_adjust=True)["Close"])
df["LR"]=np.log(df[symbol]/df[symbol].shift(1))
df

[*********************100%***********************]  1 of 1 completed


Ticker,BTC-USD,LR
Datetime,,
2026-01-19 14:30:00+00:00,92855.156250,NaN
2026-01-19 14:35:00+00:00,92681.125000,-0.001876
2026-01-19 14:40:00+00:00,92767.984375,0.000937
2026-01-19 14:45:00+00:00,92815.570312,0.000513
2026-01-19 14:50:00+00:00,92781.648438,-0.000366
...,...,...
2026-02-19 14:05:00+00:00,65890.734375,-0.000110
2026-02-19 14:10:00+00:00,66061.375000,0.002586
2026-02-19 14:15:00+00:00,65973.679688,-0.001328


**Generating Features**

In [44]:
features=["Direction","SMA",'BB','rolling_min','rolling_max','momentum','Volatility']
sma_s=7;sma_l=30;sma_window=5
df[f"{features[0]}"]=np.where(df["LR"]>0,1,0)
df[f"{features[1]}"]=df[symbol].rolling(window=sma_s).mean()-df[symbol].rolling(window=sma_l).mean()
df[f"{features[2]}"]=(df[symbol]-df[symbol].rolling(window=sma_window).mean())/df[symbol].rolling(window=sma_window).std()
df[f"{features[3]}"]=df[symbol].rolling(window=sma_window).min()/(df[symbol]-1)
df[f"{features[4]}"]=df[symbol].rolling(window=sma_window).max()/(df[symbol]-1)
df[f"{features[5]}"]=df["LR"].rolling(window=sma_window).mean()
df[f"{features[6]}"]=df["LR"].rolling(window=sma_window).std()
df.dropna(inplace=True)
df

Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility
Datetime,,,,,,,,,
2026-01-19 16:55:00+00:00,93298.664062,0.001165,1,200.012351,1.708074,0.998448,1.000011,0.000274,0.000571
2026-01-19 17:00:00+00:00,93137.859375,-0.001725,0,192.314360,-0.746831,1.000011,1.001737,-0.000107,0.001068
2026-01-19 17:05:00+00:00,93200.453125,0.000672,1,179.219940,0.067066,0.999339,1.001065,0.000102,0.001105
2026-01-19 17:10:00+00:00,93309.804688,0.001173,1,178.582254,1.111387,0.998168,1.000011,0.000334,0.001199
2026-01-19 17:15:00+00:00,93297.937500,-0.000127,0,183.224330,0.642384,0.998295,1.000138,0.000231,0.001215
...,...,...,...,...,...,...,...,...,...
2026-02-19 14:05:00+00:00,65890.734375,-0.000110,0,-396.271466,-0.089149,0.999861,1.000182,-0.000343,0.000839
2026-02-19 14:10:00+00:00,66061.375000,0.002586,1,-386.057515,1.778685,0.997278,1.000015,0.000530,0.001175
2026-02-19 14:15:00+00:00,65973.679688,-0.001328,0,-367.145982,0.427251,0.998604,1.001344,0.000218,0.001449


**Adding Lags for each features:**

In [45]:
lags=5;cols=[]
for i in features:
  for j in range(1,lags+1):
    col=f"{i}_lag_{j}"
    df[col]=df[i].shift(periods=j)
    cols.append(col)
df.dropna(inplace=True);df

Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility,Direction_lag_1,...,momentum_lag_1,momentum_lag_2,momentum_lag_3,momentum_lag_4,momentum_lag_5,Volatility_lag_1,Volatility_lag_2,Volatility_lag_3,Volatility_lag_4,Volatility_lag_5
Datetime,,,,,,,,,,,,,,,,,,,,,
2026-01-19 17:20:00+00:00,93265.500000,-0.000348,0,182.996615,0.321191,0.998642,1.000486,-0.000071,0.001108,0.0,...,0.000231,0.000334,0.000102,-0.000107,0.000274,0.001215,0.001199,0.001105,0.001068,0.000571
2026-01-19 17:25:00+00:00,93257.843750,-0.000082,0,176.378534,-0.198108,0.999395,1.000568,0.000257,0.000640,0.0,...,-0.000071,0.000231,0.000334,0.000102,-0.000107,0.001108,0.001215,0.001199,0.001105,0.001068
2026-01-19 17:30:00+00:00,93167.679688,-0.000967,0,149.130543,-1.648551,1.000011,1.001536,-0.000070,0.000779,0.0,...,0.000257,-0.000071,0.000231,0.000334,0.000102,0.000640,0.001108,0.001215,0.001199,0.001105
2026-01-19 17:35:00+00:00,93141.226562,-0.000284,0,142.521987,-1.252613,1.000011,1.001693,-0.000362,0.000356,0.0,...,-0.000070,0.000257,-0.000071,0.000231,0.000334,0.000779,0.000640,0.001108,0.001215,0.001199
2026-01-19 17:40:00+00:00,93100.703125,-0.000435,0,121.246726,-1.182663,1.000011,1.001781,-0.000423,0.000331,0.0,...,-0.000362,-0.000070,0.000257,-0.000071,0.000231,0.000356,0.000779,0.000640,0.001108,0.001215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-19 14:05:00+00:00,65890.734375,-0.000110,0,-396.271466,-0.089149,0.999861,1.000182,-0.000343,0.000839,1.0,...,-0.000719,-0.000908,-0.000855,-0.001069,-0.000894,0.001092,0.000952,0.000999,0.000804,0.000699
2026-02-19 14:10:00+00:00,66061.375000,0.002586,1,-386.057515,1.778685,0.997278,1.000015,0.000530,0.001175,0.0,...,-0.000343,-0.000719,-0.000908,-0.000855,-0.001069,0.000839,0.001092,0.000952,0.000999,0.000804
2026-02-19 14:15:00+00:00,65973.679688,-0.001328,0,-367.145982,0.427251,0.998604,1.001344,0.000218,0.001449,1.0,...,0.000530,-0.000343,-0.000719,-0.000908,-0.000855,0.001175,0.000839,0.001092,0.000952,0.000999


# **Train-Test Split data**

In [46]:
from sklearn.model_selection import train_test_split

In [47]:
x_train,x_test,y_train,y_test=train_test_split(df.iloc[:,:],df.iloc[:,0],train_size=0.75,random_state=42)
train_test=[x_train,x_test,y_train,y_test]
for i in train_test:
  print(f"shape = {i.shape}")

shape = (6669, 44)
shape = (2224, 44)
shape = (6669,)
shape = (2224,)


In [48]:
x_train.describe()

Ticker,BTC-USD,LR,Direction,SMA,BB,rolling_min,rolling_max,momentum,Volatility,Direction_lag_1,...,momentum_lag_1,momentum_lag_2,momentum_lag_3,momentum_lag_4,momentum_lag_5,Volatility_lag_1,Volatility_lag_2,Volatility_lag_3,Volatility_lag_4,Volatility_lag_5
count,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,...,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000,6669.000000
mean,77300.474720,-0.000040,0.495277,-32.261059,-0.016669,0.998270,1.001874,-0.000037,0.001583,0.497826,...,-0.000034,-0.000032,-0.000035,-0.000034,-0.000033,0.001588,0.001589,0.001590,0.001585,0.001584
std,9280.897330,0.002045,0.500015,342.552887,1.020554,0.002592,0.002857,0.000899,0.001345,0.500033,...,0.000910,0.000915,0.000902,0.000901,0.000892,0.001350,0.001355,0.001343,0.001324,0.001310
min,60514.886719,-0.019718,0.000000,-1917.024740,-1.787540,0.969636,1.000011,-0.007402,0.000038,0.000000,...,-0.009008,-0.009008,-0.009008,-0.009008,-0.007402,0.000038,0.000038,0.000038,0.000038,0.000047
25%,68809.007812,-0.000808,0.000000,-163.729204,-0.944835,0.997638,1.000015,-0.000364,0.000724,0.000000,...,-0.000371,-0.000369,-0.000368,-0.000377,-0.000381,0.000725,0.000724,0.000729,0.000726,0.000729
50%,75961.726562,-0.000010,0.000000,-12.338393,-0.006682,0.999141,1.000907,-0.000004,0.001226,0.000000,...,-0.000006,-0.000008,-0.000007,-0.000007,-0.000006,0.001225,0.001227,0.001228,0.001231,0.001233
75%,88115.296875,0.000748,1.000000,130.308780,0.899570,1.000011,1.002452,0.000348,0.001997,1.000000,...,0.000353,0.000351,0.000346,0.000346,0.000350,0.002005,0.002000,0.002006,0.002001,0.001999
max,93257.843750,0.022888,1.000000,1704.934821,1.785855,1.000017,1.040731,0.007095,0.015629,1.000000,...,0.007832,0.007832,0.007832,0.007832,0.007832,0.015629,0.015629,0.015629,0.015629,0.015627
